# Notebook概要（Bronze → Silver 変換）

このNotebookは、`bronze.Sale` の生データを分析・クレンジングし、品質チェック後に `silver.Sale` テーブルとして保存する処理です。

## 処理フロー

1. **Bronzeデータの把握**  
   - `bronze.Sale` をサンプリング取得し、欠損・ユニーク値・スキーマを確認します。

2. **プロファイリング/品質確認**  
   - 欠損値集計、数値列の統計、文字列列の分布を確認し、データ品質上の注意点を洗い出します。  
   - **補足（クレンジング観点）**: 欠損の発生率だけでなく、業務上の必須項目（主キー・日付・金額）への影響を優先して確認します。  
   - **補足（クレンジング観点）**: 分布確認では、極端値・ゼロ値・異常なカテゴリ偏りを見て、後続の変換ルール（除外/補正）を決めます。

3. **Silver向けクレンジングと標準化**  
   - 主キー重複/NULL除去、数値型キャスト、金額丸め、日付型変換を行います。  
   - `Calculated_Amount` と `Amount_Discrepancy` を使って金額整合性を検証します。  
   - `DataQualityFlag`（`Valid` / 異常理由）を付与し、`ProcessedTimestamp` を追加します。  
   - **補足（クレンジング観点）**: 型統一は分析・集計の再現性を高めるために重要で、文字列数値や日付フォーマット揺れを早い段階で吸収します。  
   - **補足（クレンジング観点）**: 金額丸めは計算差異のノイズを減らしますが、業務ルール（小数桁・丸め方法）を固定し、処理間で一貫させることが重要です。

4. **保存対象/除外対象の振り分け**  
   - `DataQualityFlag = 'Valid'` かつ `Sale_Key` が存在するレコードのみを保存対象にします。  
   - 異常レコードは除外データとして確認できます。  
   - **補足（クレンジング観点）**: 除外データは破棄せずに保持し、件数推移・原因内訳（例: `Invalid Price`）を監視すると品質改善サイクルを回しやすくなります。  
   - **補足（クレンジング観点）**: 保存/除外の判定基準は明文化し、しきい値変更時に影響比較できるよう運用します。

5. **Silverテーブルへ書き込み**  
   - クレンジング済みデータを `silver.Sale` テーブルとして上書き保存します。

6. **保存結果の確認**  
   - `demo_lakehouse.silver.Sale` を読み出して結果確認します。

## 一般的なデータクレンジングの観点（参考）

- **完全性（Completeness）**: 必須項目の欠損率、主キー欠損、参照切れを管理する。
- **妥当性（Validity）**: データ型、値域、形式（日時・コード体系）が業務ルールに合っているか確認する。
- **一意性（Uniqueness）**: 重複キーや重複イベントを検知し、重複排除ルールを固定する。
- **整合性（Consistency）**: 列間の計算関係（数量×単価=金額）や日付前後関係を検証する。
- **正確性（Accuracy）**: マスタデータや外部基準との照合で誤記・異常値を検出する。
- **追跡可能性（Auditability）**: フラグ、処理時刻、除外理由を残し、再現可能な運用にする。

## 出力データのポイント

- クレンジング済みの販売データ（型統一・丸め済み）
- 品質フラグと処理時刻を持つ監査しやすいデータ

## 本ハンズオンで実施すること

1. **セル3** のパラメータ（Lakehouse名・スキーマ名など）を自身の環境の値に変更する。
2. 先頭から **1つずつセルを実行** し、各セルの実行結果を確認する。
3. **AI関数** は書籍の手順に従って実装する（Answer用Notebookも付属しております）。
4. 後続ステップに備え、**Silverスキーマの `Sale` と `City` の結合クエリ** が実行できることを確認する。


### ▶ 実行セルの説明（環境・テーブル参照の初期設定）
**処理内容**
- Lakehouse名、Bronze/Silverスキーマ名、テーブル名を定義し、FQNと保存パスを組み立てます。

**確認ポイント**
- `LAKEHOUSE_NAME`、`BRONZE_SCHEMA`、`SILVER_SCHEMA` が自身の環境に一致していること
- 生成されたFQN/パスにスペルミスがないこと

In [ ]:
## ご自身の環境の値に変更してください
LAKEHOUSE_NAME = "demo_lakehouse" # レイクハウス名
BRONZE_SCHEMA = "bronze" # Bronzeスキーマ名
SILVER_SCHEMA = "silver" # Silverスキーマ名 

SALE_TABLE = "Sale"
CITY_TABLE = "City"

BRONZE_SALE_FQN = f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{SALE_TABLE}"
SILVER_SALE_FQN = f"{LAKEHOUSE_NAME}.{SILVER_SCHEMA}.{SALE_TABLE}"
BRONZE_CITY_FQN = f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{CITY_TABLE}"
SILVER_CITY_FQN = f"{LAKEHOUSE_NAME}.{SILVER_SCHEMA}.{CITY_TABLE}"

SILVER_SALE_PATH = f"Tables/{SILVER_SCHEMA}/{SALE_TABLE}"
SILVER_CITY_PATH = f"Tables/{SILVER_SCHEMA}/{CITY_TABLE}"

# %%sql セルから ${変数名} で参照できるよう Spark SQL 変数として登録
spark.conf.set("BRONZE_SCHEMA", "bronze")
spark.conf.set("SILVER_SCHEMA", "silver")
spark.conf.set("SALE_TABLE", "Sale")
spark.conf.set("CITY_TABLE", "City")
spark.conf.set("BRONZE_SALE_FQN", f"`{LAKEHOUSE_NAME}`.`{BRONZE_SCHEMA}`.`{SALE_TABLE}`")
spark.conf.set("SILVER_SALE_FQN", f"`{LAKEHOUSE_NAME}`.`{SILVER_SCHEMA}`.`{SALE_TABLE}`")
spark.conf.set("BRONZE_CITY_FQN", f"`{LAKEHOUSE_NAME}`.`{BRONZE_SCHEMA}`.`{CITY_TABLE}`")
spark.conf.set("SILVER_CITY_FQN", f"`{LAKEHOUSE_NAME}`.`{SILVER_SCHEMA}`.`{CITY_TABLE}`")

# 変数の内容を確認
print("LAKEHOUSE_NAME:", LAKEHOUSE_NAME)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("SALE_TABLE:", SALE_TABLE)
print("CITY_TABLE:", CITY_TABLE)
print("BRONZE_SALE_FQN:", BRONZE_SALE_FQN)
print("SILVER_SALE_FQN:", SILVER_SALE_FQN)
print("BRONZE_CITY_FQN:", BRONZE_CITY_FQN)
print("SILVER_CITY_FQN:", SILVER_CITY_FQN)
print("SILVER_SALE_PATH:", SILVER_SALE_PATH)
print("SILVER_CITY_PATH:", SILVER_CITY_PATH)

### ▶ 実行セルの説明（Sales Bronzeのプロファイリング）
**処理内容**
- Bronze `Sale` テーブルのスキーマ、欠損、基本統計、文字列分布を確認し、クレンジング方針の材料を集めます。

**確認ポイント**
- 欠損値や偏りが大きい列を特定できていること
- ID/日付/金額の候補列が検出され、後続処理に利用可能であること

In [ ]:
%%sql
-- 1. スキーマ確認
DESCRIBE ${BRONZE_SALE_FQN};

-- 2. 各列のNULL値カウント
SELECT
    COUNT(*) AS total_records,
    COUNT(*) - COUNT(Sale_Key) AS Sale_Key_nulls,
    COUNT(*) - COUNT(City_Key) AS City_Key_nulls,
    COUNT(*) - COUNT(Customer_Key) AS Customer_Key_nulls,
    COUNT(*) - COUNT(Bill_To_Customer_Key) AS Bill_To_Customer_Key_nulls,
    COUNT(*) - COUNT(Stock_Item_Key) AS Stock_Item_Key_nulls,
    COUNT(*) - COUNT(Invoice_Date_Key) AS Invoice_Date_Key_nulls,
    COUNT(*) - COUNT(Delivery_Date_Key) AS Delivery_Date_Key_nulls,
    COUNT(*) - COUNT(Salesperson_Key) AS Salesperson_Key_nulls,
    COUNT(*) - COUNT(WWI_Invoice_ID) AS WWI_Invoice_ID_nulls,
    COUNT(*) - COUNT(Description) AS Description_nulls,
    COUNT(*) - COUNT(Package) AS Package_nulls,
    COUNT(*) - COUNT(Quantity) AS Quantity_nulls,
    COUNT(*) - COUNT(Unit_Price) AS Unit_Price_nulls,
    COUNT(*) - COUNT(Tax_Rate) AS Tax_Rate_nulls,
    COUNT(*) - COUNT(Total_Excluding_Tax) AS Total_Excluding_Tax_nulls,
    COUNT(*) - COUNT(Tax_Amount) AS Tax_Amount_nulls,
    COUNT(*) - COUNT(Profit) AS Profit_nulls,
    COUNT(*) - COUNT(Total_Including_Tax) AS Total_Including_Tax_nulls,
    COUNT(*) - COUNT(Total_Dry_Items) AS Total_Dry_Items_nulls,
    COUNT(*) - COUNT(Total_Chiller_Items) AS Total_Chiller_Items_nulls
FROM ${BRONZE_SALE_FQN};

-- 3. 数値型列の基本統計量
SELECT
    'Quantity' AS column_name, MIN(Quantity) AS min_val, MAX(Quantity) AS max_val, ROUND(AVG(Quantity), 2) AS avg_val, ROUND(STDDEV(Quantity), 2) AS stddev_val FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Unit_Price', MIN(Unit_Price), MAX(Unit_Price), ROUND(AVG(Unit_Price), 2), ROUND(STDDEV(Unit_Price), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Tax_Rate', MIN(Tax_Rate), MAX(Tax_Rate), ROUND(AVG(Tax_Rate), 2), ROUND(STDDEV(Tax_Rate), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Total_Excluding_Tax', MIN(Total_Excluding_Tax), MAX(Total_Excluding_Tax), ROUND(AVG(Total_Excluding_Tax), 2), ROUND(STDDEV(Total_Excluding_Tax), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Tax_Amount', MIN(Tax_Amount), MAX(Tax_Amount), ROUND(AVG(Tax_Amount), 2), ROUND(STDDEV(Tax_Amount), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Profit', MIN(Profit), MAX(Profit), ROUND(AVG(Profit), 2), ROUND(STDDEV(Profit), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Total_Including_Tax', MIN(Total_Including_Tax), MAX(Total_Including_Tax), ROUND(AVG(Total_Including_Tax), 2), ROUND(STDDEV(Total_Including_Tax), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Total_Dry_Items', MIN(Total_Dry_Items), MAX(Total_Dry_Items), ROUND(AVG(Total_Dry_Items), 2), ROUND(STDDEV(Total_Dry_Items), 2) FROM ${BRONZE_SALE_FQN}
UNION ALL SELECT
    'Total_Chiller_Items', MIN(Total_Chiller_Items), MAX(Total_Chiller_Items), ROUND(AVG(Total_Chiller_Items), 2), ROUND(STDDEV(Total_Chiller_Items), 2) FROM ${BRONZE_SALE_FQN};

-- 4. 文字列列の分布（Description 上位10件）
SELECT Description, COUNT(*) AS count FROM ${BRONZE_SALE_FQN} GROUP BY Description ORDER BY count DESC LIMIT 10;

-- 5. 文字列列の分布（Package 上位10件）
SELECT Package, COUNT(*) AS count FROM ${BRONZE_SALE_FQN} GROUP BY Package ORDER BY count DESC LIMIT 10;

-- 6. サンプルデータ確認
SELECT * FROM ${BRONZE_SALE_FQN} LIMIT 10

### ▶ 実行セルの説明（Salesクレンジング）
**処理内容**
- 重複/NULL除去、型変換、端数処理、金額整合チェック、日付変換、品質フラグ付与を実施します。

**確認ポイント**
- `DataQualityFlag` の判定ロジックが想定どおりに機能していること
- 数値列・日付列の型変換結果と再計算差分（`Amount_Discrepancy`）を確認すること

In [ ]:
%%sql
-- Sales クレンジング（Bronze → Silver 相当の変換処理）
-- 1. 重複除去（Sale_Key でパーティション、NULL除外）
-- 2. 数値型キャスト + 端数処理
-- 3. 日付変換（in-place）
-- 4. 金額再計算・検証（Calculated_Amount, Amount_Discrepancy）
-- 5. データ品質フラグ付与
-- 6. メタデータ（ProcessedTimestamp）追加
USE ${BRONZE_SCHEMA};
CREATE OR REPLACE TEMP VIEW sales_silver AS
WITH deduplicated AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY Sale_Key ORDER BY Sale_Key) AS rn
    FROM ${SALE_TABLE}
    WHERE Sale_Key IS NOT NULL
)
SELECT
    Sale_Key, City_Key, Customer_Key, Bill_To_Customer_Key, Stock_Item_Key,
    -- 日付変換（in-place で型変換）
    TO_DATE(Invoice_Date_Key) AS Invoice_Date_Key,
    TO_DATE(Delivery_Date_Key) AS Delivery_Date_Key,
    Salesperson_Key, WWI_Invoice_ID,
    Description, Package,
    -- 数値型キャスト + 端数処理
    CAST(Quantity AS INT) AS Quantity,
    ROUND(CAST(Unit_Price AS DOUBLE), 2) AS Unit_Price,
    ROUND(CAST(Total_Excluding_Tax AS DOUBLE), 2) AS Total_Excluding_Tax,
    ROUND(CAST(Tax_Amount AS DOUBLE), 2) AS Tax_Amount,
    ROUND(CAST(Profit AS DOUBLE), 2) AS Profit,
    ROUND(CAST(Total_Including_Tax AS DOUBLE), 2) AS Total_Including_Tax,
    ROUND(CAST(Total_Dry_Items AS DOUBLE), 2) AS Total_Dry_Items,
    ROUND(CAST(Total_Chiller_Items AS DOUBLE), 2) AS Total_Chiller_Items,
    ROUND(CAST(Tax_Rate AS DOUBLE), 2) AS Tax_Rate,
    -- 金額再計算・検証
    CAST(Quantity AS INT) * ROUND(CAST(Unit_Price AS DOUBLE), 2) AS Calculated_Amount,
    ABS(ROUND(CAST(Total_Excluding_Tax AS DOUBLE), 2) - CAST(Quantity AS INT) * ROUND(CAST(Unit_Price AS DOUBLE), 2)) AS Amount_Discrepancy,
    -- データ品質フラグ
    CASE
        WHEN CAST(Quantity AS INT) <= 0 THEN 'Invalid Quantity'
        WHEN ROUND(CAST(Unit_Price AS DOUBLE), 2) <= 0 THEN 'Invalid Price'
        WHEN ABS(ROUND(CAST(Total_Excluding_Tax AS DOUBLE), 2) - CAST(Quantity AS INT) * ROUND(CAST(Unit_Price AS DOUBLE), 2)) > 0.01 THEN 'Amount Mismatch'
        ELSE 'Valid'
    END AS DataQualityFlag,
    -- メタデータ
    current_timestamp() AS ProcessedTimestamp
FROM deduplicated
WHERE rn = 1;

-- クレンジング結果の確認
SELECT * FROM sales_silver LIMIT 10;

-- スキーマ確認（データ型が変換されたことを確認）
DESCRIBE sales_silver

### ▶ 実行セルの説明（Sales保存対象の作成）
**処理内容**
- Bronze列を基準に保存カラムを選定し、`DataQualityFlag = Valid` のデータだけを保存対象として抽出します。

**確認ポイント**
- 除外列（品質チェック用の一時列）が保存対象に含まれていないこと
- フィルタ後の件数が極端に減っていないこと

In [ ]:
%%sql
-- Sales保存対象の作成（Calculated_Amount, Amount_Discrepancy を除外、Valid のみ）
CREATE OR REPLACE TEMP VIEW sales_silver_to_save AS
SELECT
    Sale_Key, City_Key, Customer_Key, Bill_To_Customer_Key, Stock_Item_Key,
    Invoice_Date_Key, Delivery_Date_Key, Salesperson_Key, WWI_Invoice_ID,
    Description, Package,
    Quantity, Unit_Price, Total_Excluding_Tax, Tax_Amount, Profit,
    Total_Including_Tax, Total_Dry_Items, Total_Chiller_Items, Tax_Rate,
    DataQualityFlag, ProcessedTimestamp
FROM sales_silver
WHERE DataQualityFlag = 'Valid'
  AND Sale_Key IS NOT NULL

### ▶ 実行セルの説明（Silverスキーマ作成）
**処理内容**
- `silver` スキーマが未作成の場合に備えて、存在チェック付きで作成します。

**確認ポイント**
- エラーなくスキーマ作成SQLが実行できること
- 既存スキーマがある場合でも安全に再実行できること

In [ ]:
%%sql
-- silver スキーマが存在しない場合、新規作成
CREATE SCHEMA IF NOT EXISTS ${SILVER_SCHEMA};

### ▶ 実行セルの説明（Silverスキーマの確認）
**処理内容**
- `DESCRIBE SCHEMA` で `silver` スキーマが認識されているか確認します。

**確認ポイント**
- スキーマ情報が正常に返ること
- スキーマ名のスペルミスがないこと

In [ ]:
%%sql
-- スキーマが認識されているか確認
DESCRIBE SCHEMA ${SILVER_SCHEMA};

### ▶ 実行セルの説明（Salesデータの保存）
**処理内容**
- 保存対象に絞り込んだSalesデータをSilver層へ `overwrite` 保存します。

**確認ポイント**
- 保存先（`SILVER_SALE_PATH`）が正しいこと
- `overwrite` による既存データ上書き影響を理解していること

In [ ]:
%%sql
-- silverスキーマ配下にSaleテーブルとして保存（上書き）
CREATE OR REPLACE TABLE ${SILVER_SALE_FQN} AS
SELECT * FROM sales_silver_to_save;

### ▶ 実行セルの説明（Sales保存結果の確認）
**処理内容**
- Silver `Sale` テーブルをサンプリング取得し、保存結果を目視確認します。

**確認ポイント**
- テーブルが正常に作成され、件数・列構成が想定どおりであること
- 品質フラグ付きで保存した内容が反映されていること

In [ ]:
%%sql
-- Silver Salesテーブルの保存結果確認
SELECT * FROM ${SILVER_SALE_FQN} LIMIT 50;

## CityテーブルのSilver化（クレンジング方針）

このセクションでは、`bronze.City` を対象に、分析しやすく監査可能な形へ標準化して `silver.City` として保存します。
`Sale` テーブルと同様に、**要件特定 → クレンジング → 品質判定 → 保存** の順で処理します。

### 実施するクレンジング内容

1. **要件特定（列プロファイリング）**  
   - 欠損/空文字の発生状況を列単位で確認し、品質課題を把握します。  
   - 主キー、必須文字列列、人口列、有効期間列の品質を確認します。

2. **標準化（型・表記ゆれの統一）**  
   - 文字列列は `TRIM` を実施し、空文字を `NULL` に統一します。  
   - 主キー列は重複排除・`NULL` 除去を行い、一意性を担保します。  
   - 人口列は `BIGINT` にキャストし、数値分析可能な型へ統一します。  
   - `Valid_From` / `Valid_To` は `TIMESTAMP` へ変換し、期間比較を可能にします。

3. **品質判定（DataQualityFlag）**  
   - 必須文字列の欠損、負の人口値、不正な期間（開始 > 終了）を検知して理由を付与します。  
   - 問題がないデータは `Valid` とし、`ProcessedTimestamp` を付与して処理時点を記録します。

4. **保存対象の制御と出力**  
   - `DataQualityFlag = 'Valid'` のデータのみを `silver.City` に保存します。  
   - 不正データは理由別集計で確認し、データ品質改善のために原因追跡できる状態を維持します。

### 出力データのポイント

- 文字列表記・型が統一された `City` マスタ
- 品質フラグと処理時刻を持つ監査しやすいデータ
- 保存対象と除外対象を明確に分離した運用可能なSilverデータ

### ▶ 実行セルの説明（City Bronzeの現状把握）
**処理内容**
- CityのBronzeテーブルを読み込み、欠損・空文字、主キー一意性、必須列、人口・期間列を分析します。

**確認ポイント**
- 各列の欠損や空文字の分布を見てクレンジング方針が妥当か確認すること
- 主キーの重複や人口・期間列の異常値がないか確認すること

In [ ]:
%%sql
-- 1. スキーマ確認
DESCRIBE ${BRONZE_CITY_FQN};

-- 2. 各列のNULL/空文字カウント
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN City_Key IS NULL OR TRIM(CAST(City_Key AS STRING)) = '' THEN 1 ELSE 0 END) AS City_Key_nulls,
    SUM(CASE WHEN WWI_City_ID IS NULL OR TRIM(CAST(WWI_City_ID AS STRING)) = '' THEN 1 ELSE 0 END) AS WWI_City_ID_nulls,
    SUM(CASE WHEN City IS NULL OR TRIM(City) = '' THEN 1 ELSE 0 END) AS City_nulls,
    SUM(CASE WHEN State_Province IS NULL OR TRIM(State_Province) = '' THEN 1 ELSE 0 END) AS State_Province_nulls,
    SUM(CASE WHEN Country IS NULL OR TRIM(Country) = '' THEN 1 ELSE 0 END) AS Country_nulls,
    SUM(CASE WHEN Continent IS NULL OR TRIM(Continent) = '' THEN 1 ELSE 0 END) AS Continent_nulls,
    SUM(CASE WHEN Sales_Territory IS NULL OR TRIM(Sales_Territory) = '' THEN 1 ELSE 0 END) AS Sales_Territory_nulls,
    SUM(CASE WHEN Region IS NULL OR TRIM(Region) = '' THEN 1 ELSE 0 END) AS Region_nulls,
    SUM(CASE WHEN Subregion IS NULL OR TRIM(Subregion) = '' THEN 1 ELSE 0 END) AS Subregion_nulls,
    SUM(CASE WHEN Latest_Recorded_Population IS NULL OR TRIM(CAST(Latest_Recorded_Population AS STRING)) = '' THEN 1 ELSE 0 END) AS Latest_Recorded_Population_nulls,
    SUM(CASE WHEN Valid_From IS NULL OR TRIM(CAST(Valid_From AS STRING)) = '' THEN 1 ELSE 0 END) AS Valid_From_nulls,
    SUM(CASE WHEN Valid_To IS NULL OR TRIM(CAST(Valid_To AS STRING)) = '' THEN 1 ELSE 0 END) AS Valid_To_nulls,
    SUM(CASE WHEN Lineage_Key IS NULL OR TRIM(CAST(Lineage_Key AS STRING)) = '' THEN 1 ELSE 0 END) AS Lineage_Key_nulls
FROM ${BRONZE_CITY_FQN};

-- 3. 文字列列の分布確認（City 上位10件）
SELECT City, COUNT(*) AS count FROM ${BRONZE_CITY_FQN} GROUP BY City ORDER BY count DESC LIMIT 10;

-- 4. 文字列列の分布確認（State_Province 上位10件）
SELECT State_Province, COUNT(*) AS count FROM ${BRONZE_CITY_FQN} GROUP BY State_Province ORDER BY count DESC LIMIT 10;

-- 5. 文字列列の分布確認（Country 上位10件）
SELECT Country, COUNT(*) AS count FROM ${BRONZE_CITY_FQN} GROUP BY Country ORDER BY count DESC LIMIT 10;

-- 6. 文字列列の分布確認（Continent 上位10件）
SELECT Continent, COUNT(*) AS count FROM ${BRONZE_CITY_FQN} GROUP BY Continent ORDER BY count DESC LIMIT 10;

-- 7. 主キーの一意性確認（City_Key のユニーク数 vs 全件数）
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT City_Key) AS unique_city_keys,
    COUNT(*) - COUNT(DISTINCT City_Key) AS duplicate_count
FROM ${BRONZE_CITY_FQN};

-- 8. 人口列の基本統計量
SELECT
    MIN(Latest_Recorded_Population) AS min_population,
    MAX(Latest_Recorded_Population) AS max_population,
    ROUND(AVG(Latest_Recorded_Population), 2) AS avg_population,
    ROUND(STDDEV(Latest_Recorded_Population), 2) AS stddev_population,
    SUM(CASE WHEN Latest_Recorded_Population < 0 THEN 1 ELSE 0 END) AS negative_count
FROM ${BRONZE_CITY_FQN};

-- 9. 期間列の確認（Valid_From / Valid_To の範囲 + 不正期間の件数）
SELECT
    MIN(Valid_From) AS earliest_valid_from,
    MAX(Valid_From) AS latest_valid_from,
    MIN(Valid_To) AS earliest_valid_to,
    MAX(Valid_To) AS latest_valid_to,
    SUM(CASE WHEN Valid_From > Valid_To THEN 1 ELSE 0 END) AS invalid_period_count
FROM ${BRONZE_CITY_FQN};

-- 10. サンプルデータ確認
SELECT * FROM ${BRONZE_CITY_FQN} LIMIT 100

### ▶ 実行セルの説明（Cityクレンジング）
**処理内容**
- 文字列の整形、主キー重複除去、型変換、妥当性チェックを行い、品質フラグと処理時刻を付与します。

**確認ポイント**
- `DataQualityFlag` が期待どおりに判定されていること
- 主キー欠損や重複が除去され、日時・人口列の型が適正なこと

In [ ]:
%%sql
-- City クレンジング（文字列trim・空文字NULL化・重複除去・型変換・品質フラグ付与）

USE ${BRONZE_SCHEMA};
CREATE OR REPLACE TEMP VIEW city_silver AS
WITH deduplicated AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY City_Key ORDER BY City_Key) AS rn
    FROM ${CITY_TABLE}
    WHERE City_Key IS NOT NULL
)
SELECT
    City_Key, WWI_City_ID,
    -- 文字列列: trim + 空文字をNULL化
    NULLIF(TRIM(City), '') AS City,
    NULLIF(TRIM(State_Province), '') AS State_Province,
    NULLIF(TRIM(Country), '') AS Country,
    NULLIF(TRIM(Continent), '') AS Continent,
    NULLIF(TRIM(Sales_Territory), '') AS Sales_Territory,
    NULLIF(TRIM(Region), '') AS Region,
    NULLIF(TRIM(Subregion), '') AS Subregion,
    -- 人口列を BIGINT にキャスト
    CAST(Latest_Recorded_Population AS BIGINT) AS Latest_Recorded_Population,
    -- 期間列を timestamp に変換
    TO_TIMESTAMP(Valid_From) AS Valid_From,
    TO_TIMESTAMP(Valid_To) AS Valid_To,
    Lineage_Key,
    -- 品質フラグ（判定順: Validity Period → Population → Continent → Country → State_Province → City）
    CASE
        WHEN TO_TIMESTAMP(Valid_From) > TO_TIMESTAMP(Valid_To) THEN 'Invalid Validity Period'
        WHEN CAST(Latest_Recorded_Population AS BIGINT) < 0 THEN 'Invalid Population'
        WHEN NULLIF(TRIM(Continent), '') IS NULL THEN 'Missing Continent'
        WHEN NULLIF(TRIM(Country), '') IS NULL THEN 'Missing Country'
        WHEN NULLIF(TRIM(State_Province), '') IS NULL THEN 'Missing State_Province'
        WHEN NULLIF(TRIM(City), '') IS NULL THEN 'Missing City'
        ELSE 'Valid'
    END AS DataQualityFlag,
    current_timestamp() AS ProcessedTimestamp
FROM deduplicated
WHERE rn = 1

### ▶ 実行セルの説明（City保存対象の作成）
**処理内容**
- Bronze列を基準に保存カラムを組み立て、`DataQualityFlag = Valid` の行だけを保存対象として抽出します。

**確認ポイント**
- 保存列に必要な項目が漏れていないこと
- 無効データが除外され、件数が想定どおりになっていること

In [ ]:
%%sql
-- City保存対象の作成（DataQualityFlag = Valid のみ）
CREATE OR REPLACE TEMP VIEW city_silver_to_save AS
SELECT
    City_Key, WWI_City_ID, City, State_Province, Country, Continent,
    Sales_Territory, Region, Subregion,
    Latest_Recorded_Population, Valid_From, Valid_To, Lineage_Key,
    DataQualityFlag, ProcessedTimestamp
FROM city_silver
WHERE DataQualityFlag = 'Valid'

### ▶ 実行セルの説明（Cityデータの保存）
**処理内容**
- クレンジング済みCityデータをSilver層に `overwrite` 保存します。

**確認ポイント**
- 保存先パス（`SILVER_CITY_PATH`）が正しいこと
- `overwrite` により既存データが置き換わる点を理解していること

In [ ]:
%%sql
-- silverスキーマ配下にCityテーブルとして保存（上書き）
CREATE OR REPLACE TABLE ${SILVER_CITY_FQN} AS
SELECT * FROM city_silver_to_save

### ▶ 実行セルの説明（City保存結果の確認）
**処理内容**
- 保存済みのSilver `City` テーブルを読み出し、サンプル表示で内容を確認します。

**確認ポイント**
- データが0件でないこと
- `DataQualityFlag` や主要属性（都市名・国名など）が正しく入っていること

In [ ]:
%%sql
-- 保存されたCityテーブルを確認
SELECT * FROM ${SILVER_CITY_FQN} LIMIT 100;

このセルのSQLは、後続の **RLS（Row-Level Security）** / **CLS（Column-Level Security）** の動作確認に利用する検証クエリです。

このクエリでは `SILVER_SALE_FQN` と `SILVER_CITY_FQN` を `City_Key` で結合し、売上明細（`Sale_Key`、`Invoice_Date_Key`、`Quantity` など）と地域情報（`City`、`State_Province`、`Country`）および金額列（`Unit_Price`、`Total_Including_Tax`）を取得します。

確認ポイントは以下です。
- **RLS確認**: 実行ユーザーの権限に応じて、参照可能な行だけが返ること（地域・組織などの条件で行が制限されること）
- **CLS確認**: 権限対象外の列が非表示/マスクされること（特に金額系列の参照可否）

同一クエリをユーザー（またはロール）を切り替えて実行し、返却行数や表示列の差分を比較して動作を確認します。

### ▶ 実行セルの説明（Sales × City 結合）
**処理内容**
- Silver層の `Sale` と `City` を `City_Key` で内部結合し、分析向けの明細データを取得します。

**確認ポイント**
- 結合キー（`City_Key`）の欠損・型不一致がないこと
- 結果件数や主要列（`City`、`Country`、金額列）が期待どおりに取得できること

In [ ]:
%%sql
-- Sales × City 結合（RLS/CLS動作確認用クエリ）
SELECT
    s.Sale_Key,
    s.City_Key,
    c.City,
    c.State_Province,
    c.Sales_Territory,
    c.Country,
    c.Latest_Recorded_Population,
    s.Invoice_Date_Key,
    s.Quantity,
    s.Unit_Price,
    s.Total_Including_Tax
FROM ${SILVER_SALE_FQN} AS s
INNER JOIN ${SILVER_CITY_FQN} AS c
    ON s.City_Key = c.City_Key
ORDER BY s.Invoice_Date_Key DESC, s.Sale_Key DESC;

### ▶ 実行セルの説明（SQLマジック）
**処理内容**
- `%%sql` マジックでSQLを直接実行するためのセルです（現在は `DROP SCHEMA` がコメントアウトされています）。

**確認ポイント**
- 破壊的操作（`DROP` など）はコメントアウト状態を維持すること
- 実行する場合は対象スキーマ名と影響範囲を必ず再確認すること

In [ ]:
%%sql
-- DROP SCHEMA silver